# Crear Google Sheets - Pruebas Atrasadas 2026

**PERIODO ESCOLAR 2026**

Desde: Miercoles 18 de marzo  
Hasta: Miercoles 9 de diciembre

---

In [ ]:
!pip install gspread oauth2client

In [1]:
import gspread
from datetime import datetime, timedelta
import time

print("Librerias cargadas OK")

Librerias cargadas OK


In [2]:
def calcular_fechas_pruebas():
    fechas = []
    fecha_actual = datetime(2026, 3, 18)  # Miercoles 18 de marzo
    fecha_fin = datetime(2026, 12, 9)     # Miercoles 9 de diciembre
    
    print(f"Periodo: {fecha_actual.strftime('%d/%m/%Y')} - {fecha_fin.strftime('%d/%m/%Y')}\n")
    
    while fecha_actual <= fecha_fin:
        # 2 = Miercoles, 5 = Sabado
        if fecha_actual.weekday() in [2, 5]:
            dia_semana = "MIERCOLES" if fecha_actual.weekday() == 2 else "SABADO"
            nombre_hoja = f"{dia_semana} {fecha_actual.day:02d}{fecha_actual.month:02d}"
            
            fechas.append({
                'nombre': nombre_hoja,
                'fecha': fecha_actual,
                'dia_semana': dia_semana
            })
        
        fecha_actual += timedelta(days=1)
    
    return fechas

fechas_2026 = calcular_fechas_pruebas()

print(f"Calculadas {len(fechas_2026)} fechas\n")
print("Primeras 10:")
for i, f in enumerate(fechas_2026[:10], 1):
    print(f"  {i}. {f['nombre']} ({f['fecha'].strftime('%d/%m/%Y')})")

print(f"\nUltimas 5:")
for i, f in enumerate(fechas_2026[-5:], len(fechas_2026)-4):
    print(f"  {i}. {f['nombre']} ({f['fecha'].strftime('%d/%m/%Y')})")

Periodo: 18/03/2026 - 09/12/2026

Calculadas 77 fechas

Primeras 10:
  1. MIERCOLES 1803 (18/03/2026)
  2. SABADO 2103 (21/03/2026)
  3. MIERCOLES 2503 (25/03/2026)
  4. SABADO 2803 (28/03/2026)
  5. MIERCOLES 0104 (01/04/2026)
  6. SABADO 0404 (04/04/2026)
  7. MIERCOLES 0804 (08/04/2026)
  8. SABADO 1104 (11/04/2026)
  9. MIERCOLES 1504 (15/04/2026)
  10. SABADO 1804 (18/04/2026)

Ultimas 5:
  73. MIERCOLES 2511 (25/11/2026)
  74. SABADO 2811 (28/11/2026)
  75. MIERCOLES 0212 (02/12/2026)
  76. SABADO 0512 (05/12/2026)
  77. MIERCOLES 0912 (09/12/2026)


In [4]:
print("Conectando a Google Sheets...\n")

try:
    gc = gspread.oauth(
        credentials_filename='credentials.json',
        authorized_user_filename='token.json'
    )
    print("Conectado exitosamente")
except FileNotFoundError:
    print("Usando OAuth directo (se abrira navegador)...")
    gc = gspread.oauth()
    print("Conectado exitosamente")
except Exception as e:
    print(f"ERROR: {e}")

Conectando a Google Sheets...

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=390120744164-niudgtqpk2hblom6o9ebstu78b1t44in.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A53823%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fspreadsheets+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&state=awWLVF5vkmr1uzpnWH9BrSBeJZrRwA&access_type=offline
Conectado exitosamente


In [7]:
titulo = "Nomina estudiantes pruebas atrasadas 2026"
spreadsheet = gc.create(titulo)

print(f"Google Sheets creado: {titulo}")
print(f"\nURL: {spreadsheet.url}")
print(f"\nID: {spreadsheet.id}")
print("\n*** GUARDA ESTE ID - lo necesitaras despues ***")

Google Sheets creado: Nomina estudiantes pruebas atrasadas 2026

URL: https://docs.google.com/spreadsheets/d/1H9b-dxrMN0Guzj69fkk-oHbgtjdAbH1TCyfKnJIWRyc

ID: 1H9b-dxrMN0Guzj69fkk-oHbgtjdAbH1TCyfKnJIWRyc

*** GUARDA ESTE ID - lo necesitaras despues ***


In [11]:
def crear_hoja_prueba(spreadsheet, nombre_hoja):
    # Crear hoja
    ws = spreadsheet.add_worksheet(title=nombre_hoja, rows=100, cols=6)
    
    # Titulo (A1:F6)
    ws.update([['PRUEBAS ATRASADAS']], 'A1')
    ws.merge_cells('A1:F6')
    
    # Formato titulo
    ws.format('A1:F6', {
        'textFormat': {'fontSize': 24, 'bold': True},
        'horizontalAlignment': 'CENTER',
        'verticalAlignment': 'MIDDLE',
        'backgroundColor': {'red': 0.6, 'green': 0.6, 'blue': 0.6}
    })
    
    # Encabezados (fila 7)
    encabezados = [
        'CICLO', 'CURSO', 'Estudiante', 'Asignatura', 'Asistencia',
        'Observaciones a entregar al alumno o reportar al docente (solo si existe alguna a informar)'
    ]
    ws.update([encabezados], 'A7:F7')
    
    # Formato encabezados
    ws.format('A7:F7', {
        'textFormat': {'bold': True},
        'horizontalAlignment': 'CENTER',
        'backgroundColor': {'red': 1, 'green': 0.9, 'blue': 0.6}
    })
    
    return ws

print("Funcion de creacion lista (VERSION CORREGIDA)")


Funcion de creacion lista (VERSION CORREGIDA)


In [12]:
print("=" * 80)
print(f"CREANDO {len(fechas_2026)} HOJAS")
print("=" * 80)
print(f"\nTiempo estimado: ~{len(fechas_2026) * 3 // 60} minutos")
print("NO interrumpas el proceso\n")

# Eliminar Sheet1
try:
    spreadsheet.del_worksheet(spreadsheet.worksheet('Sheet1'))
    print("Sheet1 eliminada\n")
except:
    pass

hojas_creadas = 0
errores = []

for i, fecha in enumerate(fechas_2026, 1):
    try:
        print(f"[{i:3d}/{len(fechas_2026)}] {fecha['nombre']:20s}", end=" ")
        crear_hoja_prueba(spreadsheet, fecha['nombre'])
        hojas_creadas += 1
        print("OK")
        time.sleep(3)
        
    except Exception as e:
        error_msg = str(e)
        print(f"ERROR: {error_msg[:40]}")
        errores.append({'nombre': fecha['nombre'], 'error': error_msg})
        
        if any(w in error_msg.lower() for w in ['quota', 'limit', 'rate']):
            print("   Esperando 60 seg...")
            time.sleep(60)
            try:
                crear_hoja_prueba(spreadsheet, fecha['nombre'])
                hojas_creadas += 1
                print("   Reintento OK")
            except:
                pass

print("\n" + "=" * 80)
print("PROCESO COMPLETADO")
print("=" * 80)
print(f"Hojas creadas: {hojas_creadas}/{len(fechas_2026)}")

if errores:
    print(f"\nErrores: {len(errores)}")
    for e in errores[:5]:
        print(f"  - {e['nombre']}")
else:
    print("\nTODAS LAS HOJAS CREADAS SIN ERRORES")

print(f"\nURL: {spreadsheet.url}")
print(f"ID: {spreadsheet.id}")

CREANDO 77 HOJAS

Tiempo estimado: ~3 minutos
NO interrumpas el proceso

[  1/77] MIERCOLES 1803       ERROR: APIError: [400]: Invalid requests[0].add
[  2/77] SABADO 2103          ERROR: APIError: [400]: Invalid requests[0].add
[  3/77] MIERCOLES 2503       ERROR: APIError: [400]: Invalid requests[0].add
[  4/77] SABADO 2803          ERROR: APIError: [400]: Invalid requests[0].add
[  5/77] MIERCOLES 0104       ERROR: APIError: [400]: Invalid requests[0].add
[  6/77] SABADO 0404          ERROR: APIError: [400]: Invalid requests[0].add
[  7/77] MIERCOLES 0804       ERROR: APIError: [400]: Invalid requests[0].add
[  8/77] SABADO 1104          ERROR: APIError: [400]: Invalid requests[0].add
[  9/77] MIERCOLES 1504       ERROR: APIError: [400]: Invalid requests[0].add
[ 10/77] SABADO 1804          ERROR: APIError: [400]: Invalid requests[0].add
[ 11/77] MIERCOLES 2204       ERROR: APIError: [400]: Invalid requests[0].add
[ 12/77] SABADO 2504          ERROR: APIError: [400]: Invalid request

In [13]:
# Ver hojas existentes
hojas_existentes = spreadsheet.worksheets()
print(f"Total de hojas: {len(hojas_existentes)}")
print("\nPrimeras 10 hojas:")
for i, h in enumerate(hojas_existentes[:10], 1):
    print(f"  {i}. {h.title}")
print("\nUltimas 10 hojas:")
for i, h in enumerate(hojas_existentes[-10:], len(hojas_existentes)-9):
    print(f"  {i}. {h.title}")

Total de hojas: 77

Primeras 10 hojas:
  1. MIERCOLES 1803
  2. SABADO 2103
  3. MIERCOLES 2503
  4. SABADO 2803
  5. MIERCOLES 0104
  6. SABADO 0404
  7. MIERCOLES 0804
  8. SABADO 1104
  9. MIERCOLES 1504
  10. SABADO 1804

Ultimas 10 hojas:
  68. SABADO 0711
  69. MIERCOLES 1111
  70. SABADO 1411
  71. MIERCOLES 1811
  72. SABADO 2111
  73. MIERCOLES 2511
  74. SABADO 2811
  75. MIERCOLES 0212
  76. SABADO 0512
  77. MIERCOLES 0912


In [ ]:
hojas = spreadsheet.worksheets()

print("=" * 80)
print("RESUMEN FINAL")
print("=" * 80)
print(f"Total: {len(hojas)} hojas")
print(f"Periodo: 18/03/2026 - 09/12/2026")

print(f"\nPrimeras 10:")
for i, h in enumerate(hojas[:10], 1):
    print(f"  {i}. {h.title}")

print(f"\nUltimas 5:")
for i, h in enumerate(hojas[-5:], len(hojas)-4):
    print(f"  {i}. {h.title}")

print("\n" + "=" * 80)
print("GOOGLE SHEETS 2026 LISTO")
print("=" * 80)

---

## COMPLETADO

**Google Sheets creado para:**
- Periodo: 18 de marzo - 9 de diciembre de 2026
- Miercoles y sabados del periodo escolar
- Estructura completa
- Listo para automatizar

**GUARDA EL ID** (mostrado arriba)

---

## SIGUIENTE PASO:

1. Copia el ID del Google Sheets
2. Abre: `Sistema_Gmail_Sheets_SIMPLE.ipynb`
3. Pega el ID en la configuracion
4. Empieza a automatizar